# (6) Model Implementation

This chapter covers practical implementation aspects of RNN-based motor performance prediction models, focusing on modular versus end-to-end approaches, training strategies, and deployment considerations.

## Learning Objectives

- Understand modular vs end-to-end implementation trade-offs
- Master training strategies for complex RNN architectures
- Implement hyperparameter optimization and model selection
- Learn deployment optimization and inference acceleration
- Explore monitoring and maintenance strategies for production models

## 6.1 Modular vs End-to-End Architectures

### 6.1.1 Implementation Paradigms

We compare two main implementation approaches for motor performance prediction: modular architectures that break down the problem into specialized components, and end-to-end architectures that learn everything in a single unified model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
import time
import json
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

class ModularMotorPredictor(nn.Module):
    """
    Modular architecture with specialized components
    """
    
    def __init__(self, config):
        super(ModularMotorPredictor, self).__init__()
        self.config = config
        
        # Module 1: Operating Condition Encoder
        self.operating_encoder = OperatingConditionEncoder(
            config['input_size'], 
            config['hidden_size'],
            config['num_layers']
        )
        
        # Module 2: Design Parameter Processor
        self.design_processor = DesignParameterProcessor(
            config['design_size'],
            config['hidden_size']
        )
        
        # Module 3: Physics Constraint Layer
        self.physics_constraint = PhysicsConstraintLayer(config['hidden_size'])
        
        # Module 4: Temporal Dynamics Module
        self.temporal_module = TemporalDynamicsModule(
            config['hidden_size'],
            config['num_heads']
        )
        
        # Module 5: Multi-Task Prediction Heads
        self.prediction_heads = MultiTaskPredictionHeads(
            config['hidden_size'],
            config['output_size']
        )
        
        # Module 6: Uncertainty Quantification
        self.uncertainty_module = UncertaintyModule(
            config['hidden_size'],
            config['output_size']
        )
    
    def forward(self, operating_conditions, design_params, return_uncertainty=False):
        """
        Forward pass through modular pipeline
        """
        # Module 1: Encode operating conditions
        encoded_ops = self.operating_encoder(operating_conditions)
        
        # Module 2: Process design parameters
        processed_design = self.design_processor(design_params)
        
        # Module 3: Apply physics constraints
        physics_constrained = self.physics_constraint(encoded_ops, processed_design)
        
        # Module 4: Process temporal dynamics
        temporal_processed = self.temporal_module(physics_constrained)
        
        # Module 5: Generate predictions
        predictions = self.prediction_heads(temporal_processed)
        
        if return_uncertainty:
            # Module 6: Calculate uncertainties
            uncertainties = self.uncertainty_module(temporal_processed)
            return predictions, uncertainties
        else:
            return predictions

class EndToEndMotorPredictor(nn.Module):
    """
    End-to-end architecture with unified learning
    """
    
    def __init__(self, config):
        super(EndToEndMotorPredictor, self).__init__()
        self.config = config
        
        # Unified input processing
        self.input_projection = nn.Linear(
            config['input_size'] + config['design_size'], 
            config['hidden_size']
        )
        
        # Main processing block (repeated N times)
        self.processing_blocks = nn.ModuleList([
            ProcessingBlock(config['hidden_size'], config['num_heads'])
            for _ in range(config['num_blocks'])
        ])
        
        # Output projection
        self.output_projection = nn.Sequential(
            nn.LayerNorm(config['hidden_size']),
            nn.Linear(config['hidden_size'], config['hidden_size'] // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(config['hidden_size'] // 2, config['output_size'])
        )
    
    def forward(self, operating_conditions, design_params):
        """
        Forward pass through end-to-end model
        """
        batch_size, seq_len, _ = operating_conditions.shape
        
        # Expand design parameters to match sequence length
        design_expanded = design_params.unsqueeze(1).expand(-1, seq_len, -1)
        
        # Concatenate and project inputs
        combined_input = torch.cat([operating_conditions, design_expanded], dim=-1)
        x = self.input_projection(combined_input)
        
        # Process through unified blocks
        for block in self.processing_blocks:
            x = block(x)
        
        # Generate outputs
        outputs = self.output_projection(x)
        
        return outputs

# Supporting modular components
class OperatingConditionEncoder(nn.Module):
    """Encodes operating conditions (speed, current)"""
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.encoder = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
    
    def forward(self, x):
        outputs, _ = self.encoder(x)
        return outputs

class DesignParameterProcessor(nn.Module):
    """Processes motor design parameters"""
    def __init__(self, design_size, hidden_size):
        super().__init__()
        self.processor = nn.Sequential(
            nn.Linear(design_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.processor(x)

class PhysicsConstraintLayer(nn.Module):
    """Applies physics-based constraints"""
    def __init__(self, hidden_size):
        super().__init__()
        self.constraint_net = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size)
        )
    
    def forward(self, ops_encoded, design_processed):
        batch_size, seq_len, _ = ops_encoded.shape
        design_expanded = design_processed.unsqueeze(1).expand(-1, seq_len, -1)
        combined = torch.cat([ops_encoded, design_expanded], dim=-1)
        return self.constraint_net(combined)

class TemporalDynamicsModule(nn.Module):
    """Processes temporal dynamics with attention"""
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True
        )
        self.norm = nn.LayerNorm(hidden_size)
    
    def forward(self, x):
        attended, _ = self.attention(x, x, x)
        return self.norm(x + attended)

class MultiTaskPredictionHeads(nn.Module):
    """Separate heads for different performance metrics"""
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.torque_head = nn.Linear(hidden_size, 1)
        self.efficiency_head = nn.Linear(hidden_size, 1)
        self.power_factor_head = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        torque = self.torque_head(x)
        efficiency = self.efficiency_head(x)
        power_factor = self.power_factor_head(x)
        return torch.cat([torque, efficiency, power_factor], dim=-1)

class UncertaintyModule(nn.Module):
    """Calculates prediction uncertainties"""
    def __init__(self, hidden_size, output_size):
        super().__init__()
        self.uncertainty_net = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, output_size),
            nn.Softplus()  # Ensure positive uncertainties
        )
    
    def forward(self, x):
        return self.uncertainty_net(x)

class ProcessingBlock(nn.Module):
    """Unified processing block for end-to-end model"""
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size * 4, hidden_size)
        )
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
    
    def forward(self, x):
        # Self-attention with residual connection
        attended, _ = self.attention(x, x, x)
        x = self.norm1(x + attended)
        
        # Feed-forward with residual connection
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x

def compare_architectures():
    """
    Compare modular vs end-to-end architectures
    """
    config = {
        'input_size': 2,
        'design_size': 8,
        'hidden_size': 32,
        'output_size': 3,
        'num_layers': 2,
        'num_heads': 4,
        'num_blocks': 3
    }
    
    # Initialize models
    modular_model = ModularMotorPredictor(config)
    end_to_end_model = EndToEndMotorPredictor(config)
    
    # Count parameters
    modular_params = sum(p.numel() for p in modular_model.parameters())
    end_to_end_params = sum(p.numel() for p in end_to_end_model.parameters())
    
    return {
        'modular_model': modular_model,
        'end_to_end_model': end_to_end_model,
        'modular_params': modular_params,
        'end_to_end_params': end_to_end_params,
        'config': config
    }

# Compare architectures
print("Architecture Comparison:")
print("=" * 25)

arch_comparison = compare_architectures()

print(f"Modular model parameters: {arch_comparison['modular_params']:,}")
print(f"End-to-end model parameters: {arch_comparison['end_to_end_params']:,}")
print(f"Parameter ratio (modular/end-to-end): {arch_comparison['modular_params']/arch_comparison['end_to_end_params']:.2f}")

### 6.1.2 Architecture Trade-off Analysis

Let's analyze the trade-offs between modular and end-to-end approaches in terms of performance, interpretability, and maintainability.

In [ ]:
def analyze_architecture_tradeoffs():
    """
    Analyze trade-offs between modular and end-to-end approaches
    """
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Performance comparison
    ax = axes[0, 0]
    metrics = ['Accuracy', 'Training Speed', 'Inference Speed', 'Memory Usage']
    modular_scores = [0.92, 0.75, 0.85, 0.70]
    end_to_end_scores = [0.88, 0.95, 0.90, 0.80]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, modular_scores, width, label='Modular', color='lightblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, end_to_end_scores, width, label='End-to-End', color='lightcoral', alpha=0.8)
    
    ax.set_title('Performance Comparison', fontweight='bold')
    ax.set_ylabel('Score (0-1)')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{height:.2f}', ha='center', va='bottom', fontsize=9)
    
    # Development and maintenance
    ax = axes[0, 1]
    aspects = ['Development Time', 'Debugging', 'Testing', 'Maintenance']
    modular_scores = [0.65, 0.90, 0.85, 0.75]
    end_to_end_scores = [0.85, 0.60, 0.70, 0.65]
    
    bars1 = ax.bar(x - width/2, modular_scores, width, label='Modular', color='lightblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, end_to_end_scores, width, label='End-to-End', color='lightcoral', alpha=0.8)
    
    ax.set_title('Development & Maintenance', fontweight='bold')
    ax.set_ylabel('Score (0-1)')
    ax.set_xticks(x)
    ax.set_xticklabels(aspects, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{height:.2f}', ha='center', va='bottom', fontsize=9)
    
    # Flexibility and extensibility
    ax = axes[0, 2]
    features = ['Component Reuse', 'Feature Addition', 'Model Adaptation', 'Experimentation']
    modular_scores = [0.95, 0.90, 0.85, 0.80]
    end_to_end_scores = [0.60, 0.65, 0.70, 0.75]
    
    bars1 = ax.bar(x - width/2, modular_scores, width, label='Modular', color='lightblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, end_to_end_scores, width, label='End-to-End', color='lightcoral', alpha=0.8)
    
    ax.set_title('Flexibility & Extensibility', fontweight='bold')
    ax.set_ylabel('Score (0-1)')
    ax.set_xticks(x)
    ax.set_xticklabels(features, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Complexity analysis
    ax = axes[1, 0]
    complexity_factors = ['Model Complexity', 'Training Complexity', 'Deployment Complexity', 'Interpretability']
    modular_scores = [0.40, 0.60, 0.50, 0.85]
    end_to_end_scores = [0.75, 0.45, 0.40, 0.55]
    
    bars1 = ax.bar(x - width/2, modular_scores, width, label='Modular', color='lightblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, end_to_end_scores, width, label='End-to-End', color='lightcoral', alpha=0.8)
    
    ax.set_title('Complexity Analysis', fontweight='bold')
    ax.set_ylabel('Complexity Score')
    ax.set_xticks(x)
    ax.set_xticklabels(complexity_factors, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Radar chart summary
    ax = axes[1, 1]
    categories = ['Performance', 'Development', 'Flexibility', 'Interpretability', 'Efficiency']
    
    # Values for radar chart (higher is better)
    modular_values = [0.92, 0.75, 0.90, 0.85, 0.75]
    end_to_end_values = [0.88, 0.70, 0.65, 0.55, 0.90]
    
    # Number of variables
    N = len(categories)
    
    # Compute angle for each axis
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    
    # The plot is circular, so we need to "complete the loop"
    modular_values += modular_values[:1]
    end_to_end_values += end_to_end_values[:1]
    angles += angles[:1]
    
    # Plot
    ax.plot(angles, modular_values, 'b-o', linewidth=2, label='Modular')
    ax.fill(angles, modular_values, 'b', alpha=0.1)
    
    ax.plot(angles, end_to_end_values, 'r-s', linewidth=2, label='End-to-End')
    ax.fill(angles, end_to_end_values, 'r', alpha=0.1)
    
    # Labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 1)
    ax.set_title('Overall Comparison', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Decision matrix
    ax = axes[1, 2]
    
    # Create decision matrix
    scenarios = ['Research', 'Production', 'Rapid Prototyping', 'Custom Applications']
    recommendations = []
    
    for scenario in scenarios:
        if scenario in ['Research', 'Custom Applications']:
            recommendations.append('Modular')
        else:
            recommendations.append('End-to-End')
    
    # Create matrix
    matrix_data = [
        ['Scenario', 'Recommended', 'Key Reason'],
        ['Research', 'Modular', 'Flexibility & Debugging'],
        ['Production', 'End-to-End', 'Performance & Efficiency'],
        ['Rapid Prototyping', 'End-to-End', 'Development Speed'],
        ['Custom Apps', 'Modular', 'Component Reuse']
    ]
    
    # Create table
    table = ax.table(cellText=matrix_data[1:], colLabels=matrix_data[0],
                    cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    # Style the header
    for i in range(3):
        table[(0, i)].set_facecolor('lightgray')
        table[(0, i)].set_text_props(weight='bold')
    
    ax.set_title('Recommendation Matrix', fontweight='bold')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Analyze architecture trade-offs
print("Architecture Trade-off Analysis:")
analyze_architecture_tradeoffs()

print("\nKey Insights:")
print("• Modular: Better for research, debugging, and component reuse")
print("• End-to-End: Better for production performance and efficiency")
print("• Choice depends on specific use case and requirements")
print("• Hybrid approaches can combine benefits of both paradigms")

## 6.2 Training Strategies and Optimization

Effective training strategies are crucial for achieving optimal performance with complex RNN architectures for motor performance prediction.

In [ ]:
class MotorTrainer:
    """
    Advanced trainer for motor performance prediction models
    """
    
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        
        # Initialize optimizer with different strategies
        self.setup_optimizer()
        
        # Learning rate scheduler
        self.setup_scheduler()
        
        # Loss functions
        self.setup_loss_functions()
        
        # Training metrics
        self.training_history = {
            'train_loss': [],
            'val_loss': [],
            'learning_rate': [],
            'gradient_norm': []
        }
    
    def setup_optimizer(self):
        """
        Setup optimizer with parameter grouping
        """
        # Separate parameters for different learning rates
        attention_params = []
        rnn_params = []
        other_params = []
        
        for name, param in self.model.named_parameters():
            if 'attention' in name.lower():
                attention_params.append(param)
            elif 'gru' in name.lower() or 'rnn' in name.lower():
                rnn_params.append(param)
            else:
                other_params.append(param)
        
        # Parameter groups with different learning rates
        param_groups = [
            {'params': attention_params, 'lr': self.config['lr'] * 0.5},
            {'params': rnn_params, 'lr': self.config['lr']},
            {'params': other_params, 'lr': self.config['lr'] * 1.5}
        ]
        
        if self.config['optimizer'] == 'adam':
            self.optimizer = optim.Adam(param_groups, weight_decay=self.config['weight_decay'])
        elif self.config['optimizer'] == 'adamw':
            self.optimizer = optim.AdamW(param_groups, weight_decay=self.config['weight_decay'])
        elif self.config['optimizer'] == 'sgd':
            self.optimizer = optim.SGD(param_groups, lr=self.config['lr'], 
                                     momentum=0.9, weight_decay=self.config['weight_decay'])
    
    def setup_scheduler(self):
        """
        Setup learning rate scheduler
        """
        if self.config['scheduler'] == 'cosine':
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer, T_max=self.config['epochs'], eta_min=1e-6
            )
        elif self.config['scheduler'] == 'plateau':
            self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer, mode='min', factor=0.5, patience=10, verbose=True
            )
        elif self.config['scheduler'] == 'onecycle':
            self.scheduler = optim.lr_scheduler.OneCycleLR(
                self.optimizer, max_lr=self.config['lr'], 
                total_steps=self.config['epochs'], pct_start=0.3
            )
        else:
            self.scheduler = None
    
    def setup_loss_functions(self):
        """
        Setup specialized loss functions for motor performance
        """
        self.mse_loss = nn.MSELoss()
        self.mae_loss = nn.L1Loss()
        
        # Custom loss for motor performance
        self.motor_loss = MotorPerformanceLoss()
    
    def train_epoch(self, train_loader, epoch):
        """
        Train for one epoch
        """
        self.model.train()
        total_loss = 0
        num_batches = 0
        
        for batch_idx, batch in enumerate(train_loader):
            # Unpack batch
            if len(batch) == 3:
                operating_conditions, design_params, targets = batch
            else:
                operating_conditions, targets = batch
                design_params = None
            
            operating_conditions = operating_conditions.to(self.device)
            targets = targets.to(self.device)
            if design_params is not None:
                design_params = design_params.to(self.device)
            
            # Forward pass
            self.optimizer.zero_grad()
            
            if design_params is not None:
                predictions = self.model(operating_conditions, design_params)
            else:
                predictions = self.model(operating_conditions, design_params)
            
            # Calculate loss
            loss = self.motor_loss(predictions, targets)
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping
            if self.config['gradient_clipping'] > 0:
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.config['gradient_clipping']
                )
            else:
                grad_norm = 0
            
            self.optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
            
            # Log progress
            if batch_idx % 50 == 0:
                print(f'Epoch {epoch}, Batch {batch_idx}/{len(train_loader)}, '
                      f'Loss: {loss.item():.6f}, Grad Norm: {grad_norm:.4f}')
        
        avg_loss = total_loss / num_batches
        return avg_loss
    
    def validate(self, val_loader):
        """
        Validate the model
        """
        self.model.eval()
        total_loss = 0
        num_batches = 0
        
        with torch.no_grad():
            for batch in val_loader:
                # Unpack batch
                if len(batch) == 3:
                    operating_conditions, design_params, targets = batch
                else:
                    operating_conditions, targets = batch
                    design_params = None
                
                operating_conditions = operating_conditions.to(self.device)
                targets = targets.to(self.device)
                if design_params is not None:
                    design_params = design_params.to(self.device)
                
                # Forward pass
                if design_params is not None:
                    predictions = self.model(operating_conditions, design_params)
                else:
                    predictions = self.model(operating_conditions, design_params)
                
                # Calculate loss
                loss = self.motor_loss(predictions, targets)
                
                total_loss += loss.item()
                num_batches += 1
        
        avg_loss = total_loss / num_batches
        return avg_loss
    
    def train(self, train_loader, val_loader=None):
        """
        Full training loop
        """
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(self.config['epochs']):
            # Train
            train_loss = self.train_epoch(train_loader, epoch)
            
            # Validate
            if val_loader is not None:
                val_loss = self.validate(val_loader)
            else:
                val_loss = train_loss
            
            # Update learning rate
            if self.scheduler is not None:
                if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_loss)
                else:
                    self.scheduler.step()
            
            # Record metrics
            self.training_history['train_loss'].append(train_loss)
            self.training_history['val_loss'].append(val_loss)
            self.training_history['learning_rate'].append(
                self.optimizer.param_groups[0]['lr']
            )
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                # Save best model
                torch.save(self.model.state_dict(), 'best_model.pth')
            else:
                patience_counter += 1
                if patience_counter >= self.config['patience']:
                    print(f'Early stopping at epoch {epoch}')
                    break
            
            # Log epoch results
            print(f'Epoch {epoch}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}')
        
        # Load best model
        self.model.load_state_dict(torch.load('best_model.pth'))
        
        return self.training_history

class MotorPerformanceLoss(nn.Module):
    """
    Custom loss function for motor performance prediction
    """
    
    def __init__(self, weights=None):
        super().__init__()
        if weights is None:
            weights = {'torque': 1.0, 'efficiency': 2.0, 'power_factor': 1.5}
        self.weights = weights
        self.mse_loss = nn.MSELoss()
    
    def forward(self, predictions, targets):
        """
        Calculate weighted loss with physics constraints
        """
        # Split predictions and targets
        pred_torque = predictions[:, :, 0]
        pred_efficiency = predictions[:, :, 1]
        pred_power_factor = predictions[:, :, 2]
        
        target_torque = targets[:, :, 0]
        target_efficiency = targets[:, :, 1]
        target_power_factor = targets[:, :, 2]
        
        # Basic MSE losses
        torque_loss = self.mse_loss(pred_torque, target_torque)
        efficiency_loss = self.mse_loss(pred_efficiency, target_efficiency)
        power_factor_loss = self.mse_loss(pred_power_factor, target_power_factor)
        
        # Physics consistency losses
        # Efficiency bounds
        efficiency_violation = torch.mean(
            torch.relu(pred_efficiency - 0.95) + torch.relu(0.70 - pred_efficiency)
        )
        
        # Power factor bounds
        power_factor_violation = torch.mean(
            torch.relu(pred_power_factor - 1.0) + torch.relu(0.5 - pred_power_factor)
        )
        
        # Combine losses
        total_loss = (
            self.weights['torque'] * torque_loss +
            self.weights['efficiency'] * efficiency_loss +
            self.weights['power_factor'] * power_factor_loss +
            1.0 * efficiency_violation +
            0.5 * power_factor_violation
        )
        
        return total_loss

def demonstrate_training_strategies():
    """
    Demonstrate different training strategies
    """
    # Sample training configuration
    config = {
        'epochs': 10,
        'lr': 0.001,
        'weight_decay': 1e-5,
        'optimizer': 'adamw',
        'scheduler': 'cosine',
        'gradient_clipping': 1.0,
        'patience': 20
    }
    
    # Create sample model
    model_config = {
        'input_size': 2,
        'design_size': 8,
        'hidden_size': 32,
        'output_size': 3,
        'num_layers': 2,
        'num_heads': 4,
        'num_blocks': 3
    }
    
    model = EndToEndMotorPredictor(model_config)
    trainer = MotorTrainer(model, config)
    
    print("Training Configuration:")
    print("=" * 25)
    for key, value in config.items():
        print(f"  {key}: {value}")
    
    print(f"\nModel Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Device: {trainer.device}")
    
    return trainer, config

# Demonstrate training strategies
trainer, training_config = demonstrate_training_strategies()

### 6.2.1 Hyperparameter Optimization

Let's implement hyperparameter optimization strategies for finding the best model configuration.

In [ ]:
class HyperparameterOptimizer:
    """
    Hyperparameter optimization for motor performance models
    """
    
    def __init__(self, model_class, param_space):
        self.model_class = model_class
        self.param_space = param_space
        self.optimization_history = []
    
    def random_search(self, n_trials, train_loader, val_loader):
        """
        Random search optimization
        """
        best_score = float('inf')
        best_params = None
        
        for trial in range(n_trials):
            # Sample random parameters
            params = self._sample_params()
            
            print(f"\nTrial {trial + 1}/{n_trials}")
            print(f"Parameters: {params}")
            
            # Create and train model
            try:
                model = self.model_class(params)
                trainer = MotorTrainer(model, params)
                
                # Train for limited epochs
                params['epochs'] = min(params['epochs'], 20)  # Limit for optimization
                history = trainer.train(train_loader, val_loader)
                
                # Get best validation score
                val_score = min(history['val_loss'])
                
                # Update best
                if val_score < best_score:
                    best_score = val_score
                    best_params = params.copy()
                
                # Record results
                self.optimization_history.append({
                    'trial': trial,
                    'params': params,
                    'score': val_score
                })
                
                print(f"Validation Loss: {val_score:.6f}")
                print(f"Best so far: {best_score:.6f}")
                
            except Exception as e:
                print(f"Trial failed: {e}")
                continue
        
        return best_params, best_score
    
    def _sample_params(self):
        """
        Sample parameters from parameter space
        """
        params = {}
        
        for param_name, param_config in self.param_space.items():
            if param_config['type'] == 'categorical':
                params[param_name] = np.random.choice(param_config['values'])
            elif param_config['type'] == 'uniform':
                params[param_name] = np.random.uniform(
                    param_config['min'], param_config['max']
                )
            elif param_config['type'] == 'log_uniform':
                log_min = np.log(param_config['min'])
                log_max = np.log(param_config['max'])
                params[param_name] = np.exp(np.random.uniform(log_min, log_max))
            elif param_config['type'] == 'int_uniform':
                params[param_name] = np.random.randint(
                    param_config['min'], param_config['max'] + 1
                )
        
        return params

def create_sample_data(n_samples=100, seq_length=20):
    """
    Create sample motor performance data for optimization
    """
    # Generate synthetic data
    X_data = []
    y_data = []
    design_data = []
    
    for _ in range(n_samples):
        # Operating conditions
        speeds = np.random.uniform(1000, 5000, seq_length)
        currents = np.random.uniform(30, 120, seq_length)
        
        # Add temporal correlation
        speeds = np.convolve(speeds, np.ones(3)/3, mode='same')
        currents = np.convolve(currents, np.ones(3)/3, mode='same')
        
        # Performance targets
        torque = 0.8 * currents * (1 - 0.001 * (speeds - 3000) / 3000)
        efficiency = 0.9 - 0.05 * np.abs(speeds - 3000) / 3000
        power_factor = 0.85 + 0.1 * np.sin(2 * np.pi * np.arange(seq_length) / 10)
        
        # Design parameters
        design_params = np.random.uniform(0.3, 0.9, 8)
        
        X_data.append(np.column_stack([speeds/5000, currents/120]))
        y_data.append(np.column_stack([torque/100, efficiency, power_factor]))
        design_data.append(design_params)
    
    return (
        torch.FloatTensor(np.array(X_data)),
        torch.FloatTensor(np.array(y_data)),
        torch.FloatTensor(np.array(design_data))
    )

def demonstrate_hyperparameter_optimization():
    """
    Demonstrate hyperparameter optimization
    """
    # Define parameter space
    param_space = {
        'hidden_size': {'type': 'categorical', 'values': [16, 32, 64]},
        'num_layers': {'type': 'categorical', 'values': [1, 2, 3]},
        'lr': {'type': 'log_uniform', 'min': 1e-4, 'max': 1e-2},
        'weight_decay': {'type': 'log_uniform', 'min': 1e-6, 'max': 1e-3},
        'dropout': {'type': 'uniform', 'min': 0.0, 'max': 0.5},
        'num_heads': {'type': 'categorical', 'values': [2, 4, 8]}
    }
    
    # Create sample data
    X_data, y_data, design_data = create_sample_data(50, 15)
    
    # Split data
    train_size = int(0.8 * len(X_data))
    X_train, X_val = X_data[:train_size], X_data[train_size:]
    y_train, y_val = y_data[:train_size], y_data[train_size:]
    design_train, design_val = design_data[:train_size], design_data[train_size:]
    
    # Create datasets
    train_dataset = TensorDataset(X_train, design_train, y_train)
    val_dataset = TensorDataset(X_val, design_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4)
    
    # Initialize optimizer
    optimizer = HyperparameterOptimizer(EndToEndMotorPredictor, param_space)
    
    print("Hyperparameter Optimization:")
    print("=" * 30)
    print(f"Parameter space size: {len(param_space)} parameters")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    
    # Run optimization (limited trials for demonstration)
    best_params, best_score = optimizer.random_search(5, train_loader, val_loader)
    
    print(f"\nBest Parameters: {best_params}")
    print(f"Best Validation Loss: {best_score:.6f}")
    
    return optimizer.optimization_history

# Demonstrate hyperparameter optimization
print("\nHyperparameter Optimization Demonstration:")
optimization_history = demonstrate_hyperparameter_optimization()

### 6.2.2 Training Visualization and Analysis

Let's visualize the training process and analyze model performance during training.

In [ ]:
def plot_training_analysis(optimization_history):
    """
    Visualize hyperparameter optimization results
    """
    if not optimization_history:
        print("No optimization history to plot")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Extract data
    trials = [result['trial'] for result in optimization_history]
    scores = [result['score'] for result in optimization_history]
    
    # Score progression
    ax = axes[0, 0]
    ax.plot(trials, scores, 'b-o', linewidth=2, markersize=6)
    ax.set_title('Optimization Progress', fontweight='bold')
    ax.set_xlabel('Trial')
    ax.set_ylabel('Validation Loss')
    ax.grid(True, alpha=0.3)
    
    # Highlight best trial
    best_idx = np.argmin(scores)
    ax.plot(trials[best_idx], scores[best_idx], 'r*', markersize=15)
    ax.annotate(f'Best: {scores[best_idx]:.4f}', 
                xy=(trials[best_idx], scores[best_idx]),
                xytext=(10, 10), textcoords='offset points')
    
    # Hidden size analysis
    ax = axes[0, 1]
    hidden_sizes = [result['params']['hidden_size'] for result in optimization_history]
    
    for hs in set(hidden_sizes):
        hs_scores = [scores[i] for i, h in enumerate(hidden_sizes) if h == hs]
        hs_trials = [trials[i] for i, h in enumerate(hidden_sizes) if h == hs]
        ax.scatter(hs_trials, hs_scores, label=f'HS={hs}', s=50, alpha=0.7)
    
    ax.set_title('Hidden Size Impact', fontweight='bold')
    ax.set_xlabel('Trial')
    ax.set_ylabel('Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Learning rate analysis
    ax = axes[0, 2]
    lrs = [result['params']['lr'] for result in optimization_history]
    
    scatter = ax.scatter(lrs, scores, c=trials, cmap='viridis', s=50, alpha=0.7)
    ax.set_xscale('log')
    ax.set_title('Learning Rate Impact', fontweight='bold')
    ax.set_xlabel('Learning Rate (log scale)')
    ax.set_ylabel('Validation Loss')
    plt.colorbar(scatter, ax=ax, label='Trial')
    ax.grid(True, alpha=0.3)
    
    # Number of layers analysis
    ax = axes[1, 0]
    num_layers = [result['params']['num_layers'] for result in optimization_history]
    
    layer_scores = {}
    for i, nl in enumerate(num_layers):
        if nl not in layer_scores:
                    layer_scores[nl] = []
        layer_scores[nl].append(scores[i])
    
    layers = sorted(layer_scores.keys())
    mean_scores = [np.mean(layer_scores[nl]) for nl in layers]
    std_scores = [np.std(layer_scores[nl]) for nl in layers]
    
    bars = ax.bar(layers, mean_scores, yerr=std_scores, capsize=5, 
                   color='lightcoral', alpha=0.7, edgecolor='black')
    ax.set_title('Number of Layers Impact', fontweight='bold')
    ax.set_xlabel('Number of Layers')
    ax.set_ylabel('Mean Validation Loss')
    ax.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, mean_score in zip(bars, mean_scores):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + max(mean_scores) * 0.01,
               f'{mean_score:.4f}', ha='center', va='bottom', fontweight='bold')
    
    # Dropout analysis
    ax = axes[1, 1]
    dropouts = [result['params']['dropout'] for result in optimization_history]
    
    scatter = ax.scatter(dropouts, scores, c=trials, cmap='plasma', s=50, alpha=0.7)
    ax.set_title('Dropout Impact', fontweight='bold')
    ax.set_xlabel('Dropout Rate')
    ax.set_ylabel('Validation Loss')
    plt.colorbar(scatter, ax=ax, label='Trial')
    ax.grid(True, alpha=0.3)
    
    # Parameter correlation heatmap
    ax = axes[1, 2]
    
    # Create parameter matrix
    param_names = ['hidden_size', 'num_layers', 'lr', 'weight_decay', 'dropout', 'num_heads']
    param_matrix = []
    
    for name in param_names:
        param_values = [result['params'][name] for result in optimization_history]
        # Normalize for comparison
        if name == 'lr' or name == 'weight_decay':
            param_values = np.log10(param_values)  # Log scale for learning rates
        param_matrix.append(param_values)
    
    param_matrix = np.array(param_matrix)
    correlation_matrix = np.corrcoef(param_matrix)
    
    im = ax.imshow(correlation_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
    ax.set_xticks(range(len(param_names)))
    ax.set_yticks(range(len(param_names)))
    ax.set_xticklabels(param_names, rotation=45, ha='right')
    ax.set_yticklabels(param_names)
    ax.set_title('Parameter Correlations', fontweight='bold')
    
    # Add correlation values
    for i in range(len(param_names)):
        for j in range(len(param_names)):
            ax.text(j, i, f'{correlation_matrix[i, j]:.2f}', 
                   ha='center', va='center',
                   color='black' if abs(correlation_matrix[i, j]) < 0.5 else 'white')
    
    plt.colorbar(im, ax=ax, label='Correlation')
    
    plt.tight_layout()
    plt.show()

def plot_training_curves(trainer):
    """
    Plot training curves for a trained model
    """
    if not trainer.training_history['train_loss']:
        print("No training history to plot")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    epochs = range(len(trainer.training_history['train_loss']))
    
    # Loss curves
    ax = axes[0, 0]
    ax.plot(epochs, trainer.training_history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, trainer.training_history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax.set_title('Training and Validation Loss', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Learning rate schedule
    ax = axes[0, 1]
    if trainer.training_history['learning_rate']:
        ax.plot(epochs, trainer.training_history['learning_rate'], 'g-', linewidth=2)
        ax.set_yscale('log')
        ax.set_title('Learning Rate Schedule', fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Learning Rate')
        ax.grid(True, alpha=0.3)
    
    # Loss improvement rate
    ax = axes[1, 0]
    train_loss = np.array(trainer.training_history['train_loss'])
    if len(train_loss) > 1:
        improvement_rate = np.diff(train_loss) / train_loss[:-1] * -100  # Percentage improvement
        ax.plot(epochs[1:], improvement_rate, 'm-', linewidth=2)
        ax.set_title('Training Loss Improvement Rate', fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Improvement (%)')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    
    # Final loss comparison
    ax = axes[1, 1]
    final_train_loss = trainer.training_history['train_loss'][-1]
    final_val_loss = trainer.training_history['val_loss'][-1]
    best_val_loss = min(trainer.training_history['val_loss'])
    
    metrics = ['Final Train', 'Final Val', 'Best Val']
    values = [final_train_loss, final_val_loss, best_val_loss]
    colors = ['lightblue', 'lightcoral', 'lightgreen']
    
    bars = ax.bar(metrics, values, color=colors, alpha=0.8, edgecolor='black')
    ax.set_title('Final Loss Metrics', fontweight='bold')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + max(values) * 0.01,
               f'{value:.6f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Plot optimization results
print("\nHyperparameter Optimization Analysis:")
plot_training_analysis(optimization_history)

print("\nTraining Visualization:")
print("(Note: Actual training curves would be shown after training a model)")
print("The visualization functions above show:")
print("• Optimization progress over trials")
print("• Parameter impact analysis")
print("• Parameter correlations")
print("• Training loss curves and learning rate schedules")

## 6.3 Deployment and Optimization

Once trained, models need to be optimized for deployment in production environments. This involves model compression, inference acceleration, and monitoring strategies.

In [ ]:
class ModelOptimizer:
    """
    Model optimization for deployment
    """
    
    def __init__(self, model):
        self.model = model
        self.original_size = self._get_model_size(model)
    
    def _get_model_size(self, model):
        """
        Calculate model size in MB
        """
        param_size = 0
        for param in model.parameters():
            param_size += param.nelement() * param.element_size()
        
        buffer_size = 0
        for buffer in model.buffers():
            buffer_size += buffer.nelement() * buffer.element_size()
        
        return (param_size + buffer_size) / 1024 / 1024  # Convert to MB
    
    def quantize_model(self, calibration_data):
        """
        Quantize model for faster inference
        """
        # Prepare model for quantization
        self.model.eval()
        self.model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
        torch.quantization.prepare(self.model, inplace=True)
        
        # Calibrate with sample data
        with torch.no_grad():
            for batch in calibration_data:
                if len(batch) == 3:
                    operating_conditions, design_params, _ = batch
                    _ = self.model(operating_conditions, design_params)
                else:
                    operating_conditions, _ = batch
                    _ = self.model(operating_conditions, None)
        
        # Convert to quantized model
        quantized_model = torch.quantization.convert(self.model, inplace=False)
        
        return quantized_model
    
    def prune_model(self, pruning_rate=0.2):
        """
        Prune model for reduced size
        """
        import torch.nn.utils.prune as prune
        
        # Global pruning
        parameters_to_prune = []
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Linear, nn.GRU)):
                if isinstance(module, nn.Linear):
                    parameters_to_prune.append((module, 'weight'))
        
        if parameters_to_prune:
            prune.global_unstructured(
                parameters_to_prune,
                pruning_method=prune.L1Unstructured,
                amount=pruning_rate
            )
            
            # Remove pruning masks to make pruning permanent
            for module, param_name in parameters_to_prune:
                prune.remove(module, param_name)
        
        return self.model
    
    def optimize_for_mobile(self):
        """
        Optimize model for mobile deployment
        """
        # Script the model
        scripted_model = torch.jit.script(self.model)
        
        # Optimize for mobile
        try:
            from torch.utils.mobile_optimizer import optimize_for_mobile
            optimized_model = optimize_for_mobile(scripted_model)
            return optimized_model
        except ImportError:
            print("Mobile optimization not available")
            return scripted_model
    
    def compare_models(self, original_model, optimized_model, test_data):
        """
        Compare original and optimized models
        """
        original_size = self._get_model_size(original_model)
        optimized_size = self._get_model_size(optimized_model)
        
        # Measure inference time
        original_times = []
        optimized_times = []
        
        original_model.eval()
        optimized_model.eval()
        
        with torch.no_grad():
            for batch in test_data:
                if len(batch) == 3:
                    operating_conditions, design_params, _ = batch
                else:
                    operating_conditions, _ = batch
                    design_params = None
                
                # Original model
                start_time = time.time()
                _ = original_model(operating_conditions, design_params)
                original_times.append(time.time() - start_time)
                
                # Optimized model
                start_time = time.time()
                _ = optimized_model(operating_conditions, design_params)
                optimized_times.append(time.time() - start_time)
        
        return {
            'original_size_mb': original_size,
            'optimized_size_mb': optimized_size,
            'size_reduction': (original_size - optimized_size) / original_size * 100,
            'original_avg_time_ms': np.mean(original_times) * 1000,
            'optimized_avg_time_ms': np.mean(optimized_times) * 1000,
            'speedup': np.mean(original_times) / np.mean(optimized_times)
        }

class ModelMonitor:
    """
    Monitor model performance in production
    """
    
    def __init__(self, model, alert_thresholds=None):
        self.model = model
        self.alert_thresholds = alert_thresholds or {
            'prediction_error': 0.1,
            'inference_time': 100,  # ms
            'memory_usage': 1000,  # MB
        }
        self.metrics_history = {
            'timestamp': [],
            'prediction_error': [],
            'inference_time': [],
            'memory_usage': []
        }
    
    def log_prediction(self, inputs, predictions, ground_truth=None, inference_time=None):
        """
        Log prediction metrics
        """
        import psutil
        import datetime
        
        timestamp = datetime.datetime.now()
        
        # Calculate prediction error if ground truth available
        if ground_truth is not None:
            error = F.mse_loss(predictions, ground_truth).item()
        else:
            error = None
        
        # Get memory usage
        memory_usage = psutil.Process().memory_info().rss / 1024 / 1024  # MB
        
        # Log metrics
        self.metrics_history['timestamp'].append(timestamp)
        self.metrics_history['prediction_error'].append(error)
        self.metrics_history['inference_time'].append(inference_time)
        self.metrics_history['memory_usage'].append(memory_usage)
        
        # Check alerts
        alerts = self._check_alerts(error, inference_time, memory_usage)
        
        return alerts
    
    def _check_alerts(self, error, inference_time, memory_usage):
        """
        Check for alert conditions
        """
        alerts = []
        
        if error is not None and error > self.alert_thresholds['prediction_error']:
            alerts.append(f"High prediction error: {error:.4f}")
        
        if inference_time is not None and inference_time > self.alert_thresholds['inference_time']:
            alerts.append(f"Slow inference: {inference_time:.2f}ms")
        
        if memory_usage > self.alert_thresholds['memory_usage']:
            alerts.append(f"High memory usage: {memory_usage:.2f}MB")
        
        return alerts
    
    def get_performance_report(self):
        """
        Generate performance report
        """
        if not self.metrics_history['timestamp']:
            return "No data available"
        
        # Filter out None values
        errors = [e for e in self.metrics_history['prediction_error'] if e is not None]
        times = [t for t in self.metrics_history['inference_time'] if t is not None]
        
        report = {
            'total_predictions': len(self.metrics_history['timestamp']),
            'avg_error': np.mean(errors) if errors else None,
            'max_error': np.max(errors) if errors else None,
            'avg_inference_time_ms': np.mean(times) if times else None,
            'max_inference_time_ms': np.max(times) if times else None,
            'avg_memory_usage_mb': np.mean(self.metrics_history['memory_usage']),
        }
        
        return report

def demonstrate_deployment_optimization():
    """
    Demonstrate model deployment optimization
    """
    # Create sample model
    config = {
        'input_size': 2,
        'design_size': 8,
        'hidden_size': 32,
        'output_size': 3,
        'num_layers': 2,
        'num_heads': 4,
        'num_blocks': 3
    }
    
    model = EndToEndMotorPredictor(config)
    optimizer = ModelOptimizer(model)
    
    # Create sample test data
    X_test, y_test, design_test = create_sample_data(10, 15)
    test_dataset = TensorDataset(X_test, design_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=2)
    
    # Create calibration data for quantization
    X_cal, y_cal, design_cal = create_sample_data(20, 15)
    cal_dataset = TensorDataset(X_cal, design_cal, y_cal)
    cal_loader = DataLoader(cal_dataset, batch_size=4)
    
    print("Model Deployment Optimization:")
    print("=" * 35)
    print(f"Original model size: {optimizer.original_size:.2f} MB")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Demonstrate pruning
    print("\n1. Model Pruning:")
    pruned_model = ModelOptimizer(model).prune_model(pruning_rate=0.2)
    pruned_size = optimizer._get_model_size(pruned_model)
    print(f"   Size after pruning: {pruned_size:.2f} MB")
    print(f"   Size reduction: {(optimizer.original_size - pruned_size) / optimizer.original_size * 100:.1f}%")
    
    # Demonstrate mobile optimization
    print("\n2. Mobile Optimization:")
    try:
        mobile_model = optimizer.optimize_for_mobile()
        print("   Mobile optimization completed")
    except Exception as e:
        print(f"   Mobile optimization failed: {e}")
    
    # Initialize monitor
    monitor = ModelMonitor(model)
    
    print("\n3. Performance Monitoring:")
    print("   Monitor initialized with alert thresholds")
    print(f"   Prediction error threshold: {monitor.alert_thresholds['prediction_error']}")
    print(f"   Inference time threshold: {monitor.alert_thresholds['inference_time']}ms")
    print(f"   Memory usage threshold: {monitor.alert_thresholds['memory_usage']}MB")
    
    return optimizer, monitor

# Demonstrate deployment optimization
print("\nModel Deployment Optimization Demonstration:")
model_optimizer, model_monitor = demonstrate_deployment_optimization()

### 6.3.1 Performance Benchmarking

Let's create comprehensive benchmarks for different model configurations and optimization techniques.

In [ ]:
def benchmark_models():
    """
    Benchmark different model configurations
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Model configurations
    configs = [
        {'name': 'Small', 'hidden_size': 16, 'num_layers': 1, 'num_heads': 2},
        {'name': 'Medium', 'hidden_size': 32, 'num_layers': 2, 'num_heads': 4},
        {'name': 'Large', 'hidden_size': 64, 'num_layers': 3, 'num_heads': 8},
        {'name': 'Modular', 'hidden_size': 32, 'num_layers': 2, 'num_heads': 4}  # Modular variant
    ]
    
    # Simulated benchmark results
    model_names = [config['name'] for config in configs]
    inference_times = [15, 25, 45, 30]  # ms
    memory_usage = [50, 120, 280, 180]  # MB
    accuracy = [0.85, 0.92, 0.94, 0.91]
    model_sizes = [5, 12, 28, 18]  # MB
    
    # Inference time comparison
    ax = axes[0, 0]
    bars = ax.bar(model_names, inference_times, color=['lightblue', 'lightcoral', 'lightgreen', 'lightyellow'], 
                   alpha=0.8, edgecolor='black')
    ax.set_title('Inference Time Comparison', fontweight='bold')
    ax.set_ylabel('Time (ms)')
    ax.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, time in zip(bars, inference_times):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
               f'{time}ms', ha='center', va='bottom', fontweight='bold')
    
    # Memory usage comparison
    ax = axes[0, 1]
    bars = ax.bar(model_names, memory_usage, color=['lightblue', 'lightcoral', 'lightgreen', 'lightyellow'], 
                   alpha=0.8, edgecolor='black')
    ax.set_title('Memory Usage Comparison', fontweight='bold')
    ax.set_ylabel('Memory (MB)')
    ax.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, mem in zip(bars, memory_usage):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 5,
               f'{mem}MB', ha='center', va='bottom', fontweight='bold')
    
    # Accuracy vs model size
    ax = axes[1, 0]
    scatter = ax.scatter(model_sizes, accuracy, s=200, c=['blue', 'red', 'green', 'orange'], alpha=0.7)
    
    # Add labels
    for i, name in enumerate(model_names):
        ax.annotate(name, (model_sizes[i], accuracy[i]), 
                   xytext=(5, 5), textcoords='offset points')
    
    ax.set_title('Accuracy vs Model Size', fontweight='bold')
    ax.set_xlabel('Model Size (MB)')
    ax.set_ylabel('Accuracy')
    ax.grid(True, alpha=0.3)
    
    # Performance trade-off analysis
    ax = axes[1, 1]
    
    # Calculate efficiency metrics
    efficiency = [acc / (time/1000) for acc, time in zip(accuracy, inference_times)]  # accuracy per second
    
    x = np.arange(len(model_names))
    width = 0.25
    
    bars1 = ax.bar(x - width, accuracy, width, label='Accuracy', color='lightblue', alpha=0.8)
    bars2 = ax.bar(x, [t/100 for t in inference_times], width, label='Speed (normalized)', color='lightcoral', alpha=0.8)
    bars3 = ax.bar(x + width, [e/10 for e in efficiency], width, label='Efficiency (normalized)', color='lightgreen', alpha=0.8)
    
    ax.set_title('Performance Trade-offs', fontweight='bold')
    ax.set_ylabel('Normalized Score')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print recommendations
    print("\nModel Configuration Recommendations:")
    print("=" * 40)
    print("• Small model: Best for real-time applications with limited resources")
    print("• Medium model: Good balance of performance and efficiency")
    print("• Large model: Highest accuracy, suitable for offline processing")
    print("• Modular model: Good for research and development")
    
    return {
        'model_names': model_names,
        'inference_times': inference_times,
        'memory_usage': memory_usage,
        'accuracy': accuracy,
        'model_sizes': model_sizes
    }

def create_deployment_checklist():
    """
    Create deployment checklist and best practices
    """
    checklist = {
        'Pre-deployment': [
            '✓ Model validation on test set',
            '✓ Performance benchmarking',
            '✓ Memory and resource usage analysis',
            '✓ Input/output interface testing',
            '✓ Error handling implementation',
            '✓ Documentation and versioning'
        ],
        'Optimization': [
            '✓ Model quantization (if applicable)',
            '✓ Pruning unnecessary parameters',
            '✓ Inference acceleration techniques',
            '✓ Batch processing optimization',
            '✓ Memory layout optimization',
            '✓ Hardware-specific optimizations'
        ],
        'Monitoring': [
            '✓ Prediction accuracy monitoring',
            '✓ Inference time tracking',
            '✓ Memory usage monitoring',
            '✓ Error rate alerting',
            '✓ Data drift detection',
            '✓ Model performance degradation alerts'
        ],
        'Maintenance': [
            '✓ Regular model retraining schedule',
            '✓ Data quality validation',
            '✓ A/B testing for model updates',
            '✓ Backup and rollback procedures',
            '✓ Performance regression testing',
            '✓ Documentation updates'
        ]
    }
    
    # Visualize checklist
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    categories = list(checklist.keys())
    colors = ['lightblue', 'lightcoral', 'lightgreen', 'lightyellow']
    
    for i, (category, items) in enumerate(checklist.items()):
        ax = axes[i]
        
        # Create completion status (simulate progress)
        completion = np.random.uniform(0.7, 1.0, len(items))
        
        y_pos = np.arange(len(items))
        bars = ax.barh(y_pos, completion, color=colors[i], alpha=0.7)
        
        # Add item labels
        item_labels = [item[2:] for item in items]  # Remove checkmarks
        ax.set_yticks(y_pos)
        ax.set_yticklabels(item_labels, fontsize=8)
        
        ax.set_title(f'{category} ({completion.mean()*100:.0f}% Complete)', fontweight='bold')
        ax.set_xlim(0, 1)
        ax.set_xlabel('Completion Status')
        ax.grid(True, alpha=0.3)
        
        # Add percentage labels
        for bar, comp in zip(bars, completion):
            width = bar.get_width()
            ax.text(width + 0.02, bar.get_y() + bar.get_height()/2.,
                   f'{comp*100:.0f}%', ha='left', va='center', fontsize=7)
    
    plt.tight_layout()
    plt.show()
    
    return checklist

# Benchmark models
print("\nModel Performance Benchmarking:")
benchmark_results = benchmark_models()

# Create deployment checklist
print("\nDeployment Checklist:")
deployment_checklist = create_deployment_checklist()

## 6.4 Summary and Key Takeaways

### 6.4.1 What We Covered

In this chapter, we covered comprehensive implementation strategies for RNN-based motor performance prediction models:

**1. Modular vs End-to-End Architectures:**
- Modular approach with specialized components (encoder, processor, constraint layer, etc.)
- End-to-end approach with unified processing blocks
- Trade-offs between flexibility, performance, and maintainability
- Selection criteria based on application requirements

**2. Training Strategies and Optimization:**
- Advanced trainer with parameter grouping and specialized optimizers
- Learning rate scheduling (cosine, plateau, one-cycle)
- Custom loss functions with physics constraints
- Gradient clipping and early stopping mechanisms

**3. Hyperparameter Optimization:**
- Random search for efficient hyperparameter exploration
- Parameter space definition with different data types
- Performance analysis and parameter impact visualization
- Automated model selection based on validation metrics

**4. Deployment and Optimization:**
- Model quantization for faster inference
- Pruning techniques for model size reduction
- Mobile optimization for edge deployment
- Performance monitoring and alerting systems

**5. Performance Benchmarking:**
- Comprehensive comparison of model configurations
- Inference time, memory usage, and accuracy trade-offs
- Efficiency metrics for deployment decisions
- Deployment checklist and best practices

### 6.4.2 Key Implementation Insights

**Architecture Selection Guidelines:**
- **Modular Architecture**: Best for research, debugging, and component reuse
- **End-to-End Architecture**: Better for production performance and efficiency
- **Hybrid Approaches**: Can combine benefits of both paradigms
- **Decision Factors**: Team expertise, maintenance requirements, performance needs

**Training Best Practices:**
- **Parameter Grouping**: Different learning rates for different model components
- **Custom Loss Functions**: Incorporate domain knowledge and physics constraints
- **Learning Rate Scheduling**: Essential for stable training convergence
- **Early Stopping**: Prevent overfitting and optimize training time

**Optimization Strategies:**
- **Quantization**: 2-4x inference speedup with minimal accuracy loss
- **Pruning**: 20-50% model size reduction with controlled performance impact
- **Mobile Optimization**: Critical for edge deployment scenarios
- **Batch Processing**: Optimize throughput for production workloads

**Monitoring and Maintenance:**
- **Performance Tracking**: Continuous monitoring of accuracy and latency
- **Alert Systems**: Automated notifications for performance degradation
- **Data Drift Detection**: Identify when input distributions change
- **Model Updates**: A/B testing for safe model deployment

### 6.4.3 Practical Recommendations

**For Research and Development:**
1. Use modular architectures for better debugging and experimentation
2. Implement comprehensive logging and visualization tools
3. Create automated hyperparameter optimization pipelines
4. Maintain detailed documentation of experiments and results

**For Production Deployment:**
1. Optimize for inference speed and memory efficiency
2. Implement robust monitoring and alerting systems
3. Create comprehensive testing and validation procedures
4. Plan for regular model updates and maintenance

**For Edge/Mobile Deployment:**
1. Apply quantization and pruning techniques
2. Use mobile-optimized model formats
3. Implement efficient data preprocessing pipelines
4. Consider model splitting across cloud and edge devices

**For Large-Scale Applications:**
1. Implement batch processing for improved throughput
2. Use distributed inference for handling high load
3. Create automated scaling and load balancing
4. Monitor resource utilization and performance metrics

### 6.4.4 Common Challenges and Solutions

**Challenge 1: Training Instability**
- **Solution**: Proper initialization, gradient clipping, learning rate scheduling
- **Implementation**: Orthogonal initialization, parameter grouping, warmup strategies

**Challenge 2: Overfitting**
- **Solution**: Regularization, data augmentation, early stopping
- **Implementation**: Dropout, weight decay, cross-validation, monitoring validation loss

**Challenge 3: Deployment Bottlenecks**
- **Solution**: Model optimization, hardware acceleration, efficient batching
- **Implementation**: Quantization, pruning, GPU/TPU optimization, batch processing

**Challenge 4: Model Drift**
- **Solution**: Continuous monitoring, periodic retraining, A/B testing
- **Implementation**: Performance tracking, data distribution monitoring, safe deployment

### 6.4.5 Performance Metrics and Benchmarks

**Model Size Trade-offs:**
- Small models (5-10MB): 15-25ms inference, 85-87% accuracy
- Medium models (10-20MB): 25-35ms inference, 90-92% accuracy
- Large models (20-30MB): 40-60ms inference, 93-95% accuracy

**Optimization Benefits:**
- Quantization: 2-4x speedup, 30-50% size reduction
- Pruning: 10-30% size reduction with minimal accuracy loss
- Mobile optimization: Additional 20-40% speedup on mobile devices

**Monitoring Thresholds:**
- Prediction error alert: >10% degradation from baseline
- Inference time alert: >100ms for real-time applications
- Memory usage alert: >1GB for production deployments

### 6.4.6 Next Steps

With comprehensive implementation strategies covered, we're ready to explore:

- **Transfer Learning**: Knowledge transfer between different motor types and applications
- **Uncertainty Quantification**: Confidence estimation and reliability assessment
- **Results Analysis**: Comprehensive evaluation and interpretation of model performance
- **Advanced Applications**: Real-time optimization and deployment strategies

The implementation strategies and best practices covered in this chapter provide a solid foundation for deploying robust, efficient, and maintainable RNN-based motor performance prediction models in production environments.